### needed libs 

In [3]:
#!pip install datasets torch numpy scikit-learn

# Step 1: Load dataset

In [5]:
from datasets import load_dataset
import pandas as pd

#load dataset
dataset = load_dataset("dair-ai/emotion")

#convert to dataframe
train_df= pd.DataFrame(dataset['train'])
test_df= pd.DataFrame(dataset['test'])

#example sample
print(train_df.head)

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

<bound method NDFrame.head of                                                     text  label
0                                i didnt feel humiliated      0
1      i can go from feeling so hopeless to so damned...      0
2       im grabbing a minute to post i feel greedy wrong      3
3      i am ever feeling nostalgic about the fireplac...      2
4                                   i am feeling grouchy      3
...                                                  ...    ...
15995  i just had a very brief time in the beanbag an...      0
15996  i am now turning and i feel pathetic that i am...      0
15997                     i feel strong and good overall      1
15998  i feel like this was such a rude comment and i...      3
15999  i know a lot but i feel so stupid because i ca...      0

[16000 rows x 2 columns]>


# Step 2: Tokenization + Padding + DataLoader

In [7]:
#!pip install torchtext

In [8]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+b\', text)

counter =Counter()
for text in dataset['train']['text']: #in the data set going through all data in train split
    tokens= tokenize(text)
    counter.update(tokens)

    
#Create word to index mapping vocabulary creation to map words to unique number starting from 2
vocab={word: idx+2 for idx, (word, _) in enumerate(counter.items())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1


#texts to tensors
def texts_to_tensors(text):
    tokens=tokenize(text)
    indices=[vocab.get(token,, vocab["<UNK>"]) for token in tokens]
    return torch.tensor(indices, dtype=torch.long)

#custom dataset
class EmotionDataset(Dataset):
    def __init__(self, split):
        self.texts = dataset[split]['text']
        self.labels = dataset[split]['label']

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return text_to_tensor(self.texts[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


#collate function for DataLoader
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=vocab["<PAD>"])
    return texts_padded, torch.stack(labels)

# Create loaders
train_loader = DataLoader(EmotionDataset('train'), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(EmotionDataset('validation'), batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(EmotionDataset('test'), batch_size=32, collate_fn=collate_fn)